In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

%matplotlib inline
from io import BytesIO

import kagglehub
from PIL import Image
from sklearn import metrics
from tensorflow.keras.layers import (
    Conv2D,
    Dense,
    Dropout,
    Flatten,
    MaxPooling2D,
)
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import plot_model
from tqdm import tqdm

path = kagglehub.dataset_download("xhlulu/140k-real-and-fake-faces")
print("Path to dataset files:", path)

base_path = os.path.join(path, "real_vs_fake", "real-vs-fake")


def ela(image_path, scale=(224, 224), quality=90):
    """
    Performs Error Level Analysis (ELA) on an image and returns a 3-channel RGB result.

    Args:
        image_path (str): Path to the image file.
        scale (tuple): Resize dimensions (width, height).
        quality (int): JPEG quality for recompression.

    Returns:
        np.ndarray: 3-channel ELA image in RGB format (uint8).
    """
    # Load and resize image
    image = Image.open(image_path).convert("RGB")
    image = image.resize(scale)

    # Save recompressed image to memory (not disk)
    buffer = BytesIO()
    image.save(buffer, format="JPEG", quality=quality)
    buffer.seek(0)

    compressed = Image.open(buffer)

    # Compute ELA
    diff = np.abs(
        np.array(image, dtype=np.int16) - np.array(compressed, dtype=np.int16)
    )
    diff = np.clip(diff * 10, 0, 255).astype(np.uint8)

    return diff


batch_size = 32
img_size = (224, 224)

ela_dir = os.path.join(base_path, "ela_images")
os.makedirs(ela_dir, exist_ok=True)

for stage in ["train", "valid", "test"]:
    stage_dir = os.path.join(base_path, stage)
    os.makedirs(stage_dir, exist_ok=True)

    for category in ["real", "fake"]:
        input_dir = os.path.join(stage_dir, category)
        output_dir = os.path.join(ela_dir, stage, category)

        # if os.path.exists(output_dir) and os.listdir(output_dir):
        #     print(f"Skipping {stage}/{category} (already processed)")
        #     continue

        os.makedirs(output_dir, exist_ok=True)

        for filename in tqdm(
            os.listdir(input_dir), desc=f"Processing {stage}/{category}"
        ):
            if filename.lower().endswith((".jpg", ".jpeg", ".png")):
                input_path = os.path.join(input_dir, filename)
                output_path = os.path.join(output_dir, filename)
                ela_image = ela(input_path, img_size)
                Image.fromarray(ela_image).save(output_path)

ela_dir = os.path.join(base_path, "ela_images")
train_dir = os.path.join(ela_dir, "train")
val_dir = os.path.join(ela_dir, "valid")
test_dir = os.path.join(ela_dir, "test")

datagen = ImageDataGenerator(
    rescale=1.0 / 255,
)

train_generator = datagen.flow_from_directory(
    train_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode="categorical",
)

val_generator = datagen.flow_from_directory(
    val_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode="categorical",
)

test_generator = datagen.flow_from_directory(
    test_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode="categorical",
    shuffle=False,
)


def build_model(input_shape=(img_size[0], img_size[1], 3)):
    model = Sequential(
        [
            Conv2D(32, (3, 3), activation="relu", input_shape=input_shape),
            MaxPooling2D((2, 2)),
            Conv2D(64, (3, 3), activation="relu"),
            MaxPooling2D((2, 2)),
            Conv2D(128, (3, 3), activation="relu"),
            MaxPooling2D((2, 2)),
            Flatten(),
            Dense(128, activation="relu"),
            Dropout(0.5),
            Dense(2, activation="softmax"),
        ]
    )
    optimizer = Adam(learning_rate=0.0001)
    model.compile(
        optimizer=optimizer,
        loss="categorical_crossentropy",
        metrics=[
            "accuracy",
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall"),
        ],
    )
    return model


model = build_model()
model.summary()

plot_model(
    model, to_file="model_architecture.png", show_shapes=True, show_layer_names=True
)

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,
)

plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 2)
plt.plot(history.history["accuracy"])
plt.plot(history.history["val_accuracy"])
plt.title("Model Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend(["train", "val"])

plt.subplot(1, 2, 1)
plt.plot(history.history["loss"])
plt.plot(history.history["val_loss"])
plt.title("model Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend(["train", "val"])
plt.show()

y_pred = model.predict(test_generator)

y_test = test_generator.classes
y_pred_labels = np.argmax(y_pred, axis=1)

Fake = False
Real = True
cm_display = metrics.ConfusionMatrixDisplay(
    confusion_matrix=metrics.confusion_matrix(y_test, y_pred_labels),
    display_labels=[Fake, Real],
)

cm_display.plot()
plt.show()

print("ROC AUC Score:", metrics.roc_auc_score(y_test, y_pred_labels))
print("AP Score:", metrics.average_precision_score(y_test, y_pred_labels))
print(metrics.classification_report(y_test, y_pred_labels))

_, accu, _, _ = model.evaluate(test_generator)
print("Final Test Acccuracy = {:.3f}".format(accu * 100))

img_path = os.path.join(ela_dir, "test", "real", "00001.jpg")
ela_image = ela(os.path.join(ela_dir, "test", "real", "00001.jpg"), img_size)
p1 = ela_image.astype(np.float32) / 255.0
p1 = np.expand_dims(p1, axis=0)
p1.shape

op = np.argmax(model.predict(p1), axis=-1)
print(op)
if op == [0]:
    print("Fake Face")
else:
    print("Real Face")